# E4 — YOLO11s Pretrained, 10 epoch, HF tự lưu/tự tiếp tục

Mỗi epoch, callback upload checkpoint đầy đủ (trước khi Ultralytics strip optimizer) lên Hugging Face. Chạy lại cùng notebook sẽ resume hoặc dừng nếu đã đủ 10 epoch.

**HF:** `iraprp/deeplearning/YOLO 11s/` — token lấy từ Kaggle Secret `HF_TOKEN2`.

**Cách dùng:** Attach ROD Final (`deeplearningg`), bật GPU + Internet + HF_TOKEN2, chọn **Save Version → Save & Run All**. Khi chạy phiên kế tiếp chỉ **Edit → Save Version → Save & Run All**, notebook tự tìm và tải checkpoint mới nhất từ HF. **Không xóa hay đổi folder** nếu muốn tiếp tục.

**Chú ý:** Phiên Kaggle mới không kế thừa `/kaggle/working`; lưu checkpoint trên HF mới cho phép tự resume. Không train trên TEST.


In [ ]:
%pip -q install 'ultralytics==8.4.165' 'huggingface_hub>=0.34,<2' 'pyyaml>=6.0'

In [ ]:
HF_FOLDER = 'YOLO 11s'
print('HF checkpoint folder:', HF_FOLDER)

In [ ]:
from pathlib import Path
import yaml
DATA_ROOT = Path('/kaggle/input/datasets/nguyenkin/deeplearningg/rod_detection_final')
if not DATA_ROOT.is_dir():
    candidates = [p for p in Path('/kaggle/input').rglob('data.yaml') if 'rod_detection_final' in str(p)]
    if len(candidates) != 1:
        raise FileNotFoundError('Không thấy ROD Final. Gắn dataset deeplearningg vào notebook. Candidate: ' + str(candidates[:5]))
    DATA_ROOT = candidates[0].parent
source_yaml = DATA_ROOT / 'data.yaml'
if not source_yaml.exists():
    raise FileNotFoundError('Thiếu data.yaml ở ' + str(DATA_ROOT))
source_cfg = yaml.safe_load(source_yaml.read_text(encoding='utf-8'))
class_names = source_cfg['names']
if isinstance(class_names, dict):
    class_names = [str(class_names[i] if i in class_names else class_names[str(i)]) for i in range(len(class_names))]
else:
    class_names = [str(x) for x in class_names]
if len(class_names) != 25:
    raise RuntimeError('Phải đúng 25 classes, hiện là: ' + str(len(class_names)))

def find_images(split, candidates):
    candidates = [str(source_cfg.get(split, ''))] + candidates
    for relative in candidates:
        if not relative:
            continue
        d = Path(relative)
        if not d.is_absolute():
            d = (DATA_ROOT / d).resolve()
        if d.is_dir() and any(d.glob('*.jpg')) or d.is_dir() and any(d.glob('*.png')) or d.is_dir() and any(d.glob('*.jpeg')):
            return d
    raise FileNotFoundError(f'Không tìm được images cho split {split} tại {DATA_ROOT}')
TRAIN_IMAGES = find_images('train', ['train/images', 'images/train'])
VAL_IMAGES = find_images('val', ['valid/images', 'val/images', 'images/val', 'images/valid'])
TEST_IMAGES = None  # TEST giữ kín, KHÔNG truy cập khi train.
working_yaml = Path('/kaggle/working/rod_trainval.yaml')
working_yaml.write_text(yaml.safe_dump({
    'path': str(DATA_ROOT),
    'train': str(TRAIN_IMAGES),
    'val': str(VAL_IMAGES),
    'nc': 25,
    'names': class_names,
}, allow_unicode=True, sort_keys=False), encoding='utf-8')
print('Dataset:', DATA_ROOT, '\nTRAIN:', TRAIN_IMAGES, '\nVAL:', VAL_IMAGES, '\nClasses:', len(class_names))

EXPECTED_ROD_NAMES = ['Bike', 'Building', 'Car', 'Person', 'Stairs', 'Traffic sign', 'Electrical Pole', 'Road', 'Motorcycle', 'Dustbin', 'Dog', 'Manhole', 'Tree', 'Guard rail', 'Pedestrian crosswalk', 'Truck', 'Bus', 'Bench', 'Traffic Cone', 'Fire hydrant', 'Teraffic Barrel', 'Plant Pot', 'Electrical Box', 'Chair', 'Bicycle Rack']

if class_names != EXPECTED_ROD_NAMES or source_cfg.get('nc') != 25:
    raise RuntimeError('ROD Final class ID/order không khớp bản freeze')
for split_name, images_dir, expected_count in [('TRAIN', TRAIN_IMAGES, 18151), ('VALID', VAL_IMAGES, 3318)]:
    image_files = [p for p in images_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png', '.webp')]
    labels_dir = images_dir.parent / 'labels'
    if len(image_files) != expected_count or not labels_dir.is_dir():
        raise RuntimeError(f'{split_name} image/label directory count không khớp ROD Final A4')
    image_stems = {p.stem for p in image_files}
    label_stems = {p.stem for p in labels_dir.glob('*.txt')}
    if image_stems != label_stems:
        raise RuntimeError(f'{split_name} image/label không khớp từng file')
print('ROD Final A4 TRAIN/VALID image-label pairs và 25 class: PASS')


In [ ]:
"""Embedded in both Kaggle notebooks. No local import is required at runtime."""

import hashlib
import json
import os
import shutil
import time
import uuid
from pathlib import Path


HF_REPO_ID = "iraprp/deeplearning"
HF_SECRET_NAME = "HF_TOKEN2"


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for part in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")).hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False, sort_keys=True), encoding="utf-8")
    os.replace(temp, path)


def validate_progress(manifest, folder, model, config_hash, dataset_hash, max_epoch=10):
    if manifest.get("schema") != 2 or manifest.get("folder") != folder or manifest.get("model") != model:
        raise RuntimeError("HF manifest thuộc mô hình/folder khác hoặc sai schema")
    if manifest.get("config_sha256") != config_hash or manifest.get("dataset_sha256") != dataset_hash:
        raise RuntimeError("HF checkpoint khác config/dataset; dừng, không train lại")
    epoch = manifest.get("completed_epoch")
    if type(epoch) is not int or not 0 <= epoch <= max_epoch:
        raise RuntimeError("HF completed_epoch không hợp lệ")
    if manifest.get("is_complete") is not (epoch == max_epoch):
        raise RuntimeError("HF trạng thái COMPLETE không khớp epoch")
    if not isinstance(manifest.get("sha256"), dict) or not manifest["sha256"]:
        raise RuntimeError("HF manifest thiếu checksum")
    return epoch


class HFStore:
    """Immutable snapshot first, latest pointer second. Failed upload cannot advance latest."""

    def __init__(self, api, token, repo_type, folder, model, config_hash, dataset_hash, work):
        assert folder in ("YOLO 11s", "Faster R-CNN")
        self.api, self.token, self.repo_type = api, token, repo_type
        self.folder, self.model = folder, model
        self.config_hash, self.dataset_hash = config_hash, dataset_hash
        self.work = Path(work)
        self.work.mkdir(parents=True, exist_ok=True)

    def _files(self):
        return set(self.api.list_repo_files(repo_id=HF_REPO_ID, repo_type=self.repo_type, token=self.token))

    def _download(self, remote):
        from huggingface_hub import hf_hub_download
        return Path(hf_hub_download(repo_id=HF_REPO_ID, repo_type=self.repo_type,
                                    filename=remote, token=self.token, local_dir=str(self.work / "download"),
                                    force_download=remote.endswith("latest.json")))

    def restore(self):
        files = self._files()  # connection/auth errors must propagate
        prefix = self.folder + "/"
        relevant = [name for name in files if name.startswith(prefix)]
        pointer_name = prefix + "latest.json"
        if pointer_name not in files:
            if relevant:
                raise RuntimeError("HF folder có dữ liệu nhưng thiếu latest.json; dừng để kiểm tra thủ công")
            return None
        pointer = json.loads(self._download(pointer_name).read_text(encoding="utf-8"))
        snap = pointer.get("snapshot")
        if not isinstance(snap, str) or not snap.startswith(prefix + "snapshots/") or ".." in snap:
            raise RuntimeError("HF latest pointer không hợp lệ")
        manifest_name = snap + "/manifest.json"
        if manifest_name not in files:
            raise RuntimeError("HF latest pointer trỏ tới snapshot thiếu manifest")
        manifest_path = self._download(manifest_name)
        if sha256_file(manifest_path) != pointer.get("manifest_sha256"):
            raise RuntimeError("HF manifest SHA256 không khớp latest pointer")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        validate_progress(manifest, self.folder, self.model, self.config_hash, self.dataset_hash)
        if manifest.get("snapshot") != snap:
            raise RuntimeError("HF snapshot ID không nhất quán")
        local = {}
        for name, expected in manifest["sha256"].items():
            if name != Path(name).name or snap + "/" + name not in files:
                raise RuntimeError("HF snapshot thiếu artifact hoặc tên file không hợp lệ")
            path = self._download(snap + "/" + name)
            if sha256_file(path) != expected:
                raise RuntimeError("HF artifact SHA256 không khớp: " + name)
            local[name] = path
        return manifest, local

    def _retry(self, action):
        for attempt in range(3):
            try:
                return action()
            except Exception as exc:
                print("HF upload lỗi", attempt + 1, type(exc).__name__, "; bản local còn ở", self.work)
                if attempt == 2:
                    raise RuntimeError("HF upload thất bại; giữ Kaggle Output và KHÔNG đánh dấu đã đồng bộ") from None
                time.sleep(5 * (attempt + 1))

    def save(self, files, completed_epoch, extra=None):
        snap = self.folder + "/snapshots/" + uuid.uuid4().hex
        stage = self.work / "snapshots" / snap.rsplit("/", 1)[-1]
        stage.mkdir(parents=True, exist_ok=False)
        hashes = {}
        for name, source in files.items():
            if name != Path(name).name or not Path(source).is_file():
                raise RuntimeError("Artifact không hợp lệ: " + name)
            target = stage / name
            # Kaggle checkpoint and stage live on the same filesystem. Hardlinks
            # avoid duplicating a large optimizer checkpoint during HF upload.
            # Fall back to a copy if the filesystem does not support hardlinks.
            try:
                os.link(Path(source).resolve(), target)
            except OSError:
                shutil.copy2(source, target)
            hashes[name] = sha256_file(target)
        manifest = {**(extra or {}), "schema": 2, "folder": self.folder, "model": self.model,
                    "snapshot": snap, "config_sha256": self.config_hash,
                    "dataset_sha256": self.dataset_hash,
                    "completed_epoch": int(completed_epoch), "is_complete": int(completed_epoch) == 10,
                    "sha256": hashes}
        validate_progress(manifest, self.folder, self.model, self.config_hash, self.dataset_hash)
        atomic_json(stage / "manifest.json", manifest)
        self._retry(lambda: self.api.upload_folder(folder_path=str(stage), path_in_repo=snap,
                     repo_id=HF_REPO_ID, repo_type=self.repo_type, token=self.token,
                     commit_message=f"{self.folder}: snapshot epoch {completed_epoch}"))
        # Verify server snapshot before moving pointer. A failed pointer commit leaves previous version valid.
        remote_manifest = self._download(snap + "/manifest.json")
        if sha256_file(remote_manifest) != sha256_file(stage / "manifest.json"):
            raise RuntimeError("HF upload manifest verification failed")
        pointer = self.work / "latest.json"
        atomic_json(pointer, {"snapshot": snap, "manifest_sha256": sha256_file(stage / "manifest.json")})
        self._retry(lambda: self.api.upload_file(path_or_fileobj=str(pointer), path_in_repo=self.folder + "/latest.json",
                    repo_id=HF_REPO_ID, repo_type=self.repo_type, token=self.token,
                    commit_message=f"{self.folder}: advance epoch {completed_epoch}"))
        remote_pointer = json.loads(self._download(self.folder + "/latest.json").read_text(encoding="utf-8"))
        if remote_pointer != json.loads(pointer.read_text(encoding="utf-8")):
            raise RuntimeError("HF latest pointer chưa được xác nhận sau upload")
        # Only temporary staging files are removed; the source checkpoint in
        # Kaggle Output and the verified immutable HF snapshot remain intact.
        if stage.resolve().parent != (self.work / "snapshots").resolve():
            raise RuntimeError("Local staging path outside expected snapshots directory")
        for staged_file in stage.iterdir():
            if not staged_file.is_file():
                raise RuntimeError("Unexpected directory in local staging snapshot")
            staged_file.unlink()
        stage.rmdir()
        print("HF snapshot verified:", self.folder, "epoch", completed_epoch,
              "next_sample", manifest.get("next_sample", 0), "local staging cleared; free_GB=",
              round(shutil.disk_usage(self.work).free / (1024 ** 3), 2))
        return manifest


def connect_hf(token):
    from huggingface_hub import HfApi
    if not token or not token.strip():
        raise RuntimeError("Thiếu Kaggle Secret HF_TOKEN2")
    api = HfApi(token=token)
    matches = []
    for kind in ("dataset", "model"):
        try:
            api.repo_info(repo_id=HF_REPO_ID, repo_type=kind)
            matches.append(kind)
        except Exception:
            pass
    if len(matches) != 1:
        raise RuntimeError("Không xác định duy nhất repo_type của iraprp/deeplearning; kiểm tra token/quyền/repo")
    return api, matches[0]


In [ ]:
FROZEN_E4_YAML = "# E4 main budget: stop after 10 completed epochs of an Ultralytics 30-epoch LR schedule.\n# This matches E3's actual first 10 epochs; do not describe E3 as a 10-epoch schedule.\nexperiment_id: E4_yolo11s_pretrained\nmain_budget_epochs: 10\ntask: detect\nmode: train\nmodel: yolo11s.pt\ndata: /kaggle/working/rod_e4_yolo11s/rod_trainval.yaml\npretrained: true\nepochs: 30\nbatch: 8\nimgsz: 640\noptimizer: SGD\nlr0: 0.01\nlrf: 0.01\nmomentum: 0.937\nweight_decay: 0.0005\nwarmup_epochs: 3.0\nwarmup_momentum: 0.8\nwarmup_bias_lr: 0.1\nbox: 7.5\ncls: 0.5\ndfl: 1.5\nhsv_h: 0.015\nhsv_s: 0.7\nhsv_v: 0.4\ndegrees: 0.0\ntranslate: 0.1\nscale: 0.5\nshear: 0.0\nperspective: 0.0\nflipud: 0.0\nfliplr: 0.5\nbgr: 0.0\nmosaic: 1.0\nmixup: 0.0\ncutmix: 0.0\ncopy_paste: 0.0\nclose_mosaic: 10\nseed: 0\ndeterministic: true\npatience: 30\namp: true\nrect: false\ncache: false\nworkers: 2\nfraction: 1.0\nval: true\nplots: true\nsave: true\nsave_period: 5\ndevice: 0\nproject: /kaggle/working/rod_e4_yolo11s/main\nname: E4_yolo11s_pretrained\nexist_ok: false\n"

"""Standalone E4 notebook training cell (embedded by build_hf_autoresume_notebooks.py)."""

import json
import os
import shutil
import time
from pathlib import Path

import torch
import ultralytics
import yaml
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient


E4_FOLDER = "YOLO 11s"
E4_MODEL = "yolo11s.pt"
E4_BUDGET = 10
E4_DEADLINE_HOURS = 10.0
E4_OUTPUT = Path("/kaggle/working/rod_e4_yolo11s_hf")
E4_OUTPUT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("E4 cần Kaggle GPU; chưa có CUDA")
if ultralytics.__version__ != "8.4.165":
    raise RuntimeError("E4 cần Ultralytics 8.4.165 để xác minh callback/checkpoint")

# FROZEN_E4_YAML is embedded from configs/e4_yolo11s_pretrained.yaml by the builder.
frozen = yaml.safe_load(FROZEN_E4_YAML)
if frozen["model"] != E4_MODEL or frozen["main_budget_epochs"] != 10 or frozen["epochs"] != 30:
    raise RuntimeError("Cấu hình E4 đã bị thay đổi ngoài thiết kế đã kiểm toán")

train_args = {k: v for k, v in frozen.items() if k not in ("experiment_id", "main_budget_epochs", "task", "mode")}
train_args.update(data=str(working_yaml), project=str(E4_OUTPUT), name="train", exist_ok=True,
                  model=E4_MODEL, device=0)
config_identity = dict(train_args)
config_identity["data"] = "ROD_FINAL_A4_TRAIN_VALID"
config_identity["project"] = "E4_OUTPUT"
config_identity.pop("exist_ok", None)
config_identity["ultralytics_version"] = ultralytics.__version__
config_hash = fingerprint(config_identity)
dataset_hash = sha256_file(source_yaml)
print("E4 config SHA256:", config_hash, "dataset YAML SHA256:", dataset_hash)
atomic_json(E4_OUTPUT / "config.json", config_identity)
atomic_json(E4_OUTPUT / "environment.json", {"torch": torch.__version__,
                                            "ultralytics": ultralytics.__version__,
                                            "gpu": torch.cuda.get_device_name(0)})

hf_token = UserSecretsClient().get_secret(HF_SECRET_NAME)
hf_api, repo_type = connect_hf(hf_token)
store = HFStore(hf_api, hf_token, repo_type, E4_FOLDER, E4_MODEL, config_hash,
                dataset_hash, E4_OUTPUT / "hf_backup")
restored = store.restore()
deadline = time.monotonic() + E4_DEADLINE_HOURS * 3600


def inspect_full_yolo_checkpoint(path, expected_completed=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ckpt, dict) or not isinstance(ckpt.get("optimizer"), dict):
        raise RuntimeError("E4 checkpoint thiếu optimizer; không thể resume")
    if not ckpt["optimizer"].get("param_groups") or "state" not in ckpt["optimizer"]:
        raise RuntimeError("E4 optimizer state không đầy đủ")
    if not isinstance(ckpt.get("scaler"), dict) or ckpt.get("ema") is None:
        raise RuntimeError("E4 checkpoint thiếu AMP scaler hoặc EMA")
    completed = int(ckpt.get("epoch", -1)) + 1
    if not 1 <= completed <= E4_BUDGET:
        raise RuntimeError("E4 checkpoint epoch ngoài ngân sách")
    if expected_completed is not None and completed != expected_completed:
        raise RuntimeError("E4 checkpoint epoch khác HF manifest")
    args = ckpt.get("train_args", {})
    for key in ("epochs", "imgsz", "batch", "optimizer", "lr0", "lrf", "warmup_epochs",
                "momentum", "weight_decay", "close_mosaic", "seed", "amp"):
        if args.get(key) != frozen[key]:
            raise RuntimeError("E4 effective train args không khớp cấu hình: " + key)
    ema = ckpt["ema"]
    if getattr(ema, "nc", None) != 25 or getattr(ema, "yaml", {}).get("scale") != "s":
        raise RuntimeError("E4 checkpoint architecture hoặc class count không khớp")
    names = getattr(ema, "names", {})
    ordered = [names[i] if i in names else names[str(i)] for i in range(25)] if isinstance(names, dict) else list(names)
    if ordered != class_names:
        raise RuntimeError("E4 checkpoint class names/order không khớp ROD Final")
    return completed


if restored:
    state, recovered = restored
    if "recovery_full.pt" not in recovered:
        raise RuntimeError("E4 HF snapshot thiếu recovery_full.pt")
    completed = inspect_full_yolo_checkpoint(recovered["recovery_full.pt"], state["completed_epoch"])
    resume_file = E4_OUTPUT / "train" / "weights" / "last.pt"
    resume_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(recovered["recovery_full.pt"], resume_file)
    for name in ("results.csv", "args.yaml"):
        if name not in recovered:
            raise RuntimeError("E4 HF snapshot thiếu " + name)
        shutil.copy2(recovered[name], E4_OUTPUT / "train" / name)
    if "best.pt" in recovered:
        shutil.copy2(recovered["best.pt"], resume_file.parent / "best.pt")
else:
    completed, resume_file = 0, None

if completed == E4_BUDGET:
    print("E4 COMPLETED 10/10 theo manifest và full checkpoint; chỉ hậu kiểm, không train")
else:
    model = YOLO(str(resume_file) if resume_file else E4_MODEL)
    if not resume_file and len(model.names) != 80:
        raise RuntimeError("yolo11s.pt không phải COCO pretrained 80 class")

    def backup_epoch(trainer):
        epoch_done = int(trainer.epoch) + 1
        full = Path(trainer.last)
        inspect_full_yolo_checkpoint(full, epoch_done)
        # Keep a local full-state copy before final_eval strips last.pt.
        local_full = E4_OUTPUT / "recovery_full.pt"
        temp = E4_OUTPUT / "recovery_full.pt.tmp"
        shutil.copy2(full, temp)
        os.replace(temp, local_full)
        files = {"recovery_full.pt": local_full,
                 "last.pt": local_full,
                 "results.csv": Path(trainer.csv),
                 "args.yaml": Path(trainer.save_dir) / "args.yaml",
                 "config.json": E4_OUTPUT / "config.json",
                 "environment.json": E4_OUTPUT / "environment.json"}
        if Path(trainer.best).is_file():
            files["best.pt"] = Path(trainer.best)
        manifest = store.save(files, epoch_done, extra={"architecture": "YOLO11s", "class_count": 25,
                                                       "learning_rate_schedule_epochs": 30})
        atomic_json(E4_OUTPUT / "last_hf_manifest.json", manifest)
        if epoch_done >= E4_BUDGET or time.monotonic() >= deadline:
            trainer.stop = True
            print("E4 dừng an toàn sau epoch", epoch_done, "; HF snapshot đã xác minh")

    model.add_callback("on_model_save", backup_epoch)
    if resume_file:
        # 30-epoch scheduler remains in checkpoint. No restart from epoch 1.
        model.train(resume=True, data=str(working_yaml), device=0)
    else:
        model.train(**{k: v for k, v in train_args.items() if k != "model"})

latest = store.restore()
if latest is None:
    raise RuntimeError("E4 không có HF checkpoint sau phiên này")
final_state, final_files = latest
inspect_full_yolo_checkpoint(final_files["recovery_full.pt"], final_state["completed_epoch"])
print("E4 status:", final_state["completed_epoch"], "/10", "COMPLETE" if final_state["is_complete"] else "CONTINUE")


## Xác nhận kết quả
Mở `last_hf_manifest.json` trong Kaggle Output và `latest.json` trong đúng HF folder. Chỉ coi COMPLETE khi `completed_epoch=10`, checksum và hậu kiểm checkpoint PASS. Nếu upload lỗi, lấy `hf_backup/`, checkpoint và log trong Kaggle Output; phiên sau không được âm thầm bắt đầu lại epoch 1.
